# 04 · Thinking divergence (W2) — internal state vs surfaced reply

ShareChat-Claude `thinking` as a partial reference for positive divergences (silent assumption, false confidence, reasoning↔surface mismatch).
**Backs:** W2 section of `docs/shared-findings.md`, `docs/communication/w1-codebook.md` (12% / 18%).
**Script:** `thinking_divergence.compute(judge="heuristic")` (live) + the saved `data/derived/thinking_divergence.json` (LLM-judge n=50, no LLM re-run).

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT/"src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT/"src"))
import pandas as pd
pd.set_option("display.max_rows", 120)
print("repo root:", ROOT)

repo root: /data/wang/junh/githubs/human-agent-coupling-errors


## Heuristic judge (lexical rules, no LLM) — runs live
Doc claim (heuristic column): assumption present-internally-only **0.6%**; internal-doubt/surface-confidence **18.6%** (overcounts); ambiguity-noticed-no-question **0.5%**. Mismatch is left to the LLM judge (n/a here).

In [2]:
import thinking_divergence as td
r = td.compute(judge="heuristic")
print(f"reasoning turns: {r['n_pairs']} across {r['n_convs']} convs | n_reasoning judged: {r['n_reasoning']}")
pd.DataFrame([{"signal": k, "heuristic_pct": round(v, 1)} for k, v in r["rates"].items()])

reasoning turns: 1001 across 254 convs | n_reasoning judged: 1001


,signal,heuristic_pct
0,silent_assumption,0.6
1,false_confidence,18.6
2,reasoning_surface_mismatch,0.0
3,ack_ambiguity_not_clarified,0.5


## LLM judge (Claude CLI, n=50) — loaded from saved verdicts
Doc claim: `silent_assumption` **12%** (6/50), `reasoning_surface_mismatch` **18%** (9/50), `false_confidence` **2%** (1/50). These come from the cached `thinking_divergence.json` so no LLM call is made.

In [3]:
j = json.loads((ROOT/"data"/"derived"/"thinking_divergence.json").read_text())
n = j["n_reasoning"]; counts = j["counts"]
print(f"judge={j['judge']}  n_reasoning={n}")
sigs = ["silent_assumption","false_confidence","reasoning_surface_mismatch","ack_ambiguity_not_clarified"]
pd.DataFrame([{"signal": s, "count": counts.get(s, 0),
               "llm_judge_pct": round(counts.get(s, 0)/n*100, 1)} for s in sigs])

judge=claude  n_reasoning=50


,signal,count,llm_judge_pct
0,silent_assumption,6,12.0
1,false_confidence,1,2.0
2,reasoning_surface_mismatch,9,18.0
3,ack_ambiguity_not_clarified,0,0.0


**Reading:** the heuristic and LLM-judge columns disagree by design — the lexical heuristic over-counts `false_confidence` (any hedge-word mismatch) and cannot see `reasoning_surface_mismatch`, which is the divergence the internal trace makes newly detectable (18%).